# Data Cleaning

In [65]:
# import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim

## 1. Reading parquet files

In [66]:
# define paths to parquet files

forenames = "../data/bronze/df_forenames/"
countries = "../data/bronze/df_countryCode/"

In [67]:
# set a spark session

def create_spark_session(app_name:str):
    spark = SparkSession.builder \
        .appName(app_name) \
        .master("local[*]") \
        .getOrCreate()
    return spark

In [68]:
# Create Spark session for silver stage

spark = create_spark_session("silver_pipeline")

In [69]:
df_fn = spark.read.parquet(forenames)
df_fn.show(5)

+------------+------+-------+-----+
|    forename|gender|country|count|
+------------+------+-------+-----+
|       Yasif|  NULL|     LY|    2|
| رحومه رحومه|  NULL|     LY|    2|
| رحومة رحومة|  NULL|     LY|    2|
|          Nä|  NULL|     LY|    2|
|العمر الضياع|  NULL|     LY|    2|
+------------+------+-------+-----+
only showing top 5 rows


In [70]:
df_fn.printSchema()

root
 |-- forename: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- count: integer (nullable = true)



In [71]:
df_cc = spark.read.parquet(countries)
df_cc.show(5)

+------------+--------------------+
|country_code|        country_name|
+------------+--------------------+
|          AE|United Arab Emirates|
|          AF|         Afghanistan|
|          AL|             Albania|
|          AO|              Angola|
|          AR|           Argentina|
+------------+--------------------+
only showing top 5 rows


In [72]:
df_cc.printSchema()

root
 |-- country_code: string (nullable = true)
 |-- country_name: string (nullable = true)



## 2. Data Exploration 

In [73]:
df_fn.forename.isNull()

Column<'isNull(forename)'>

In [74]:
# function to show null values per column in a specific dataframe

def count_null_values(df):
    for c in df.columns:
        print(c,": ", df.filter(col(c).isNull()).count())

In [75]:
# Shownull values for forename dataframe
count_null_values(df_fn)

forename :  291
gender :  2419135
country :  27572
count :  0


In [76]:
df_fn.count()

12470920

In [77]:
# Shownull values for country code dataframe
count_null_values(df_cc)

country_code :  0
country_name :  0


In [78]:
# show the number of records stored on country code dataframe
df_cc.count()

105

## 2. Data Cleaning

In [79]:
# drop null values on forenames dataframe
df_fn_cleaned = df_fn.dropna()
count_null_values(df_fn_cleaned)

forename :  0
gender :  0
country :  0
count :  0


In [80]:
# Start to work with country code dataframe to normalice country names
"""
Conver country name to lower case and remove blank spaces at star and at the end of the name
"""
df_cc = df_cc.withColumn('country_name', lower(col('country_name')))
df_cc = df_cc.withColumn('country_name', trim(col('country_name')))
df_cc.show(200)

+------------+--------------------+
|country_code|        country_name|
+------------+--------------------+
|          AE|united arab emirates|
|          AF|         afghanistan|
|          AL|             albania|
|          AO|              angola|
|          AR|           argentina|
|          AT|             austria|
|          AZ|          azerbaijan|
|          BD|          bangladesh|
|          BE|             belgium|
|          BF|        burkina faso|
|          BG|            bulgaria|
|          BH|             bahrain|
|          BI|             burundi|
|          BN|              brunei|
|          BO|             bolivia|
|          BR|              brazil|
|          BW|            botswana|
|          CA|              canada|
|          CH|         switzerland|
|          CL|               chile|
|          CM|            cameroon|
|          CN|               china|
|          CO|            colombia|
|          CR|          costa rica|
|          CY|              

In [81]:
# Performing a join between forenames and country_codes
"""Once we are sure that forenames and county_codes don´t have null values,
We can proceed withe table joining, we will use a left join to ensure that
we will be able to see if any value on forenames doesn't has a match on country_code
and manage it"""

df_result = df_fn_cleaned.join(df_cc, df_fn_cleaned['country'] == df_cc['country_code'], 'left')
df_result.show(5)

+--------+------+-------+-----+------------+------------+
|forename|gender|country|count|country_code|country_name|
+--------+------+-------+-----+------------+------------+
|      Md|     M|     LY|    2|          LY|       libya|
|    None|     M|     MA|20691|          MA|     morocco|
|    None|     F|     MA| 7141|          MA|     morocco|
|    Da*I|     M|     MA| 3945|          MA|     morocco|
|      De|     M|     MA|  618|          MA|     morocco|
+--------+------+-------+-----+------------+------------+
only showing top 5 rows


In [82]:
# Show if there are any null value after left join

count_null_values(df_result)

forename :  0
gender :  0
country :  0
count :  0
country_code :  0
country_name :  0


## 3. Store cleaned dataframes

In [83]:
df_fn_cleaned.write.mode("overwrite").parquet("../data/silver/df_forenames_cleaned/")
df_cc.write.mode("overwrite").parquet("../data/silver/df_countryCode_cleaned/")
df_result.write.mode("overwrite").parquet("../data/silver/df_join_fn_cc_cleaned/")